In [11]:
#IMPORTS

import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# from itertools import combinations
from scipy.optimize import curve_fit
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [12]:
#LOADING CONSTANTS

FPS = 120
LEN_ERR_MM = 0.5
GEOM = json.load(open("geometry.json"))

In [13]:
mm, g = 1e-3, 1e-3 #mm -> m, g -> kg

#positions (m), relative to each arm's local origin
JOINT1 = GEOM["arm1"]["joint"] * mm
PIVOT1 = JOINT2 = 0.0
BLUE = GEOM["arm1"]["sticker_centre"] * mm
RED = GEOM["arm2"]["sticker_centre"] * mm

TOP1 = -GEOM["arm1"]["pivot_top"] * mm
BOTTOM1 = (GEOM["arm1"]["length"] - GEOM["arm1"]["pivot_top"]) * mm
TOP2 = -GEOM["arm2"]["pivot_top"] * mm
BOTTOM2 = (GEOM["arm2"]["length"] - GEOM["arm2"]["pivot_top"]) * mm

# Masses [kg]
M1 = (GEOM["arm1"]["mass"] + GEOM["arm1"].get("mass_pivot", 0) + GEOM["arm1"].get("mass_joint", 0)) * g
M2 = GEOM["arm2"]["mass"] * g

# Assembly centres of mass [m]
COM1 = (GEOM["arm1"]["mass"] * g * (GEOM["arm1"]["length"]/2 - GEOM["arm1"]["pivot_top"])*mm + GEOM["arm1"].get("mass_joint", 0)*g*JOINT1)/M1
COM2 = (GEOM["arm2"]["length"]/2 - GEOM["arm2"]["pivot_top"]) * mm

# Moments of inertia about local origins [kg m^2]
I1_O = (
    GEOM["arm1"]["mass"]*g *
        ((GEOM["arm1"]["length"]*mm)**2
         + (GEOM["arm1"]["width"]*mm)**2)/12
    + GEOM["arm1"]["mass"]*g*(
        (GEOM["arm1"]["length"]/2 - GEOM["arm1"]["pivot_top"])*mm
    )**2
    + GEOM["arm1"].get("mass_pivot", 0)*g *
        (GEOM["arm1"].get("size_pivot", 0)*mm)**2/6
    + GEOM["arm1"].get("mass_joint", 0)*g * (
        (GEOM["arm1"].get("diameter_joint", 0)*mm/2)**2
        + JOINT1**2
    )
)

I2_O = (M2 * ((GEOM["arm2"]["length"]*mm)**2 + (GEOM["arm2"]["width"]*mm)**2)/12 + M2 * COM2**2)

# Moments of inertia about assembly COMs [kg m^2]
I1_C = I1_O - M1*COM1**2
I2_C = I2_O - M2*COM2**2

In [14]:
DATA_DIR = Path("data")
runs = {}
for f in sorted(DATA_DIR.glob("*.csv")):
    data = np.genfromtxt(f, dtype=float, delimiter=",", skip_header=1, missing_values="", filling_values=np.nan)

    parts = f.stem.replace("_data", "").split("_")
    init, run = int(parts[1]), int(parts[2])
    t=data[:, 1]
    x_red_px, y_red_px = 1920 - data[:, 2], 1080 - data[:, 3]
    x_blue_px, y_blue_px = 1920 - data[:, 4], 1080 - data[:, 5]
    x_hinge_px, y_hinge_px = np.nanmedian(1920 - data[:, 6]), np.nanmedian(1080 - data[:, 7])

    pixel_distance = np.hypot(x_blue_px - x_hinge_px, y_blue_px - y_hinge_px)
    ppm = np.nanmean(pixel_distance) / BLUE

    #take the hinge as the origin in the new coordinate system
    hinge = np.zeros(2)

    joint = np.column_stack(((x_blue_px - x_hinge_px) / ppm * JOINT1 / BLUE, (y_blue_px - y_hinge_px) / ppm * JOINT1 / BLUE))

    com1 = np.column_stack(((x_blue_px - x_hinge_px) / ppm * COM1 / BLUE, (y_blue_px - y_hinge_px) / ppm * COM1 / BLUE))

    com2 = joint + (np.column_stack((x_red_px, y_red_px)) / ppm - np.array([x_hinge_px, y_hinge_px]) / ppm - joint) * (COM2 / RED)

    runs[(init, run)] = {
        "ppm": ppm,
        "t": data[:, 1],
        "hinge": hinge, #fixed position, shape (2,)
        "joint": joint, #shape (n_frames, 2)
        "com1": com1, #shape (n_frames, 2)
        "com2": com2, #shape (n_frames, 2)
    }

print(len(runs), "runs loaded")

23 runs loaded


In [15]:
def fit_circle(x, y):
    """Returns (cx, cy, r), their errors, residuals in px (NaN where input is NaN), angle of each point."""
    ok = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[ok], y[ok]
    f = lambda xy, cx, cy, r: np.hypot(xy[0] - cx, xy[1] - cy) - r
    p0 = [xm.mean(), ym.mean(), np.hypot(xm - xm.mean(), ym - ym.mean()).mean()]
    p, cov = curve_fit(f, (xm, ym), np.zeros_like(xm), p0=p0)
    res = np.full(len(x), np.nan)
    res[ok] = f((xm, ym), *p)
    return p, np.sqrt(np.diag(cov)), res, np.arctan2(y - p[1], x - p[0])

def coverage(phi):
    return np.mean(np.histogram(phi[np.isfinite(phi)], bins=36, range=(-np.pi, np.pi))[0] > 0)

def harmonics(phi, res, n=4):
    """residual = sum of cos/sin(k*angle). k=1 centre error, k=2 ellipse (tilt), k>=3 lens/tracking."""
    ok = np.isfinite(res)
    A = np.column_stack([np.ones(ok.sum())] + [fn(k * phi[ok]) for k in range(1, n + 1) for fn in (np.cos, np.sin)])
    coef = np.linalg.lstsq(A, res[ok], rcond=None)[0]
    model = np.full(len(res), np.nan)
    model[ok] = A @ coef
    return model, np.array([np.hypot(coef[2 * k - 1], coef[2 * k]) for k in range(1, n + 1)])

def harmonic_report(runs, key, title, verbose=True):
    phi = np.concatenate([r[f"phi_{key}"] for r in runs])
    res = np.concatenate([r[f"res_{key}"] for r in runs])
    model, amps = harmonics(phi, res)
    ok = np.isfinite(res)
    noise = np.std((res - model)[ok])
    if verbose:
        print(f"{title}: harmonics k=1..4 = {amps.round(3)} px (rough 1-sigma {noise*np.sqrt(2/ok.sum()):.3f}); random noise left = {noise:.3f} px")
    return phi, res, model, noise

def plot_bias(ax, runs, key, title):
    phi, res, model, noise = harmonic_report(runs, key, title)
    edges = np.linspace(-np.pi, np.pi, 37)
    idx = np.digitize(phi, edges) - 1
    mid = np.degrees((edges[:-1] + edges[1:]) / 2)
    m = np.array([np.nanmean(res[idx == i]) if np.any(idx == i) else np.nan for i in range(36)])
    s = np.array([np.nanstd(res[idx == i]) / np.sqrt(max((idx == i).sum(), 1)) for i in range(36)])
    ax.errorbar(mid, m, s, fmt="o", ms=3)
    ok = np.isfinite(model); o = np.argsort(phi[ok])
    ax.plot(np.degrees(phi[ok][o]), model[ok][o], "r-")
    ax.axhline(0, color="k", lw=0.5)
    ax.set(title=title, xlabel="angle around fitted centre (deg)", ylabel="radial residual (px)")

def fit_ellipse(x, y):
    """Rotated ellipse. q = minor/major axis ratio; tilt = arccos(q); alpha = direction of the major axis."""
    ok = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[ok], y[ok]
    c0 = np.array([xm.mean(), ym.mean()])
    w, V = np.linalg.eigh(np.cov(np.vstack([xm - c0[0], ym - c0[1]])))
    alpha0 = np.arctan2(V[1, 1], V[0, 1])
    q0 = float(np.clip(np.sqrt(w[0] / w[1]), 0.8, 0.999))
    def f(xy, cx, cy, a, q, al):
        dx, dy = xy[0] - cx, xy[1] - cy
        return np.hypot(dx * np.cos(al) + dy * np.sin(al), (-dx * np.sin(al) + dy * np.cos(al)) / q) - a
    p, cov = curve_fit(f, (xm, ym), np.zeros_like(xm), p0=[*c0, np.sqrt(2 * w[1]), q0, alpha0])
    err = np.sqrt(np.diag(cov))
    cx, cy, a, q, al = p
    if q > 1:
        a, q, al = a * q, 1 / q, al + np.pi / 2
    al = (al + np.pi / 2) % np.pi - np.pi / 2
    return dict(c=(cx, cy), q=q, alpha=al, alpha_deg=np.degrees(al), tilt_deg=np.degrees(np.arccos(min(q, 1.0))),
                tilt_err_deg=np.degrees(err[3] / max(np.sqrt(1 - q**2), 1e-6)))

def tilt_matrix(q, alpha):
    c, s = np.cos(alpha), np.sin(alpha)
    R = np.array([[c, -s], [s, c]])
    return R @ np.diag([1, 1 / q]) @ R.T

def smooth(y, n=5):
    return np.convolve(np.pad(y, (n // 2, n - 1 - n // 2), mode="edge"), np.ones(n) / n, mode="valid")